In [130]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def plot(title, label, x, result, expected):
    plt.plot(x, expected, label=f'Ground truth')
    plt.plot(x, result, label=f'Sequre {label}')
    plt.title(title)
    plt.xlabel('x')
    plt.ylabel('y')
    plt.legend()
    plt.grid(True)
    plt.show()

def mae(result, expected):
    return np.mean(np.abs(np.array(result) - np.array(expected)))

In [131]:
""" Core methods """

# Until Codon Jupyter is fixed: Read the data from files
show_plots = False

dump_folder = "dump"
dump_files = [
    "decor_trig",
    "decor",
    "fourier",
    "cheby",
    "taylor"
    ]
nbit_fs = [32]
intervals_count = 4
cps = [0, 1]

df_data = {
    'Method': [],
    'Interval': [],
    'MAE': [],
    'Runtime': [],
    'Partitions count': [],
    'Truncations count': []
    }

for cp in cps:
    df_data[f'Bytes sent CP{cp}'] = []
    df_data[f'Requests sent CP{cp}'] = []

for dump_file in dump_files:
    for nbit_f in nbit_fs:
        for i in range(intervals_count):
            for cp in cps:
                try:
                    with open(f"{dump_folder}/{dump_file}_{i}_{nbit_f}_CP{cp}.p", "rb") as f:
                        data = pickle.load(f)
                        x = data['x']
                        interval = f"({round(min(x), 2)}, {round(max(x), 2)})"
                        for k, v in data.items():
                            if not k.endswith('_result'):
                                continue
                            
                            k = k.replace('_result', '')
                            expected = data[f"{k}_expected"]

                            runtime = round(data[f"{k}_time"][0], 5)
                            bytes_sent = int(data[f"{k}_bytes_sent"][0])
                            send_requests = int(data[f"{k}_send_requests"][0])
                            partitions_count = int(data[f"{k}_partitions_count"][0])
                            truncations_count = int(data[f"{k}_truncations_count"][0])
                            
                            if cp == 1:
                                df_data['Method'].append(f"{k}_{nbit_f}")
                                df_data['Interval'].append(interval)
                                df_data['MAE'].append(mae(v, expected))
                                
                                df_data['Runtime'].append(runtime)
                                df_data['Partitions count'].append(partitions_count)
                                df_data['Truncations count'].append(truncations_count)

                            df_data[f'Bytes sent CP{cp}'].append(bytes_sent)
                            df_data[f'Requests sent CP{cp}'].append(send_requests)
                            
                            if show_plots and cp == 1:
                                plot(f"{k} {nbit_f} frac bits on {interval}", k, x, v, expected)
                except FileNotFoundError:
                    print(f"Could not find {dump_folder}/{dump_file}_{i}_{nbit_f}.p")

df = pd.DataFrame(df_data)

In [132]:
def by_interval(df):
    for interval, group in df.groupby('Interval'):
        display(group)

In [133]:
by_interval(df[df['Method'].str.contains('sin')].sort_values(by='MAE'))

,Method,Interval,MAE,Runtime,Partitions count,Truncations count,Bytes sent CP0,Requests sent CP0,Bytes sent CP1,Requests sent CP1
3,decor_sin_32,"(-1.0, 2.5)",5.807584e-10,0.00078,2,1,9664,8,4832,4
70,chebyshev_sin_naive_32,"(-1.0, 2.5)",7.256694e-07,0.00031,8,10,67664,56,43496,36
72,chebyshev_sin_clenshaw_32,"(-1.0, 2.5)",7.257346e-07,0.00158,8,10,67648,56,43488,36
66,chebyshev_sin_decor_32,"(-1.0, 2.5)",7.260070e-07,0.00208,24,4,62152,50,36624,58
133,taylor_sin_decor_32,"(-1.0, 2.5)",9.445949e-04,0.00214,24,3,52504,46,34208,56
128,taylor_sin_naive_32,"(-1.0, 2.5)",9.445958e-04,0.00142,6,7,48336,40,31416,26
30,fourier_sin_32,"(-1.0, 2.5)",5.198747e-02,0.00098,2,1,96384,8,26512,4


,Method,Interval,MAE,Runtime,Partitions count,Truncations count,Bytes sent CP0,Requests sent CP0,Bytes sent CP1,Requests sent CP1
25,fourier_sin_32,"(-3.14, 3.14)",5.143223e-10,0.00631,2,1,9696,8,4840,4
1,decor_sin_32,"(-3.14, 3.14)",5.172606e-10,0.00049,2,1,9664,8,4832,4
57,chebyshev_sin_clenshaw_32,"(-3.14, 3.14)",1.632013e-04,0.00031,8,10,67648,56,43488,36
55,chebyshev_sin_naive_32,"(-3.14, 3.14)",1.632013e-04,0.00029,8,10,67664,56,43496,36
51,chebyshev_sin_decor_32,"(-3.14, 3.14)",1.632033e-04,0.00241,24,4,62152,50,36624,58
118,taylor_sin_naive_32,"(-3.14, 3.14)",7.196224e-02,0.00087,6,7,48336,40,31416,26
123,taylor_sin_decor_32,"(-3.14, 3.14)",7.196225e-02,0.00155,24,3,52504,46,34208,56


,Method,Interval,MAE,Runtime,Partitions count,Truncations count,Bytes sent CP0,Requests sent CP0,Bytes sent CP1,Requests sent CP1
7,decor_sin_32,"(-5.0, 5.0)",5.522537e-10,0.00006,2,1,9664,8,4832,4
111,chebyshev_sin_clenshaw_32,"(-5.0, 5.0)",7.381257e-03,0.00121,8,10,67648,56,43488,36
109,chebyshev_sin_naive_32,"(-5.0, 5.0)",7.381257e-03,0.00030,8,10,67664,56,43496,36
105,chebyshev_sin_decor_32,"(-5.0, 5.0)",7.381306e-03,0.00243,24,4,62152,50,36624,58
43,fourier_sin_32,"(-5.0, 5.0)",6.993225e-02,0.00078,2,1,96384,8,26512,4
159,taylor_sin_decor_32,"(-5.0, 5.0)",1.587863e+00,0.00216,24,3,52504,46,34208,56
154,taylor_sin_naive_32,"(-5.0, 5.0)",1.587863e+00,0.00107,6,7,48336,40,31416,26


,Method,Interval,MAE,Runtime,Partitions count,Truncations count,Bytes sent CP0,Requests sent CP0,Bytes sent CP1,Requests sent CP1
92,chebyshev_sin_naive_32,"(0.79, 2.36)",1.456224e-10,0.00029,8,10,67664,56,43496,36
5,decor_sin_32,"(0.79, 2.36)",3.586251e-10,0.00006,2,1,9664,8,4832,4
95,chebyshev_sin_clenshaw_32,"(0.79, 2.36)",3.757675e-10,0.00031,8,10,67648,56,43488,36
83,chebyshev_sin_decor_32,"(0.79, 2.36)",1.888621e-09,0.00232,24,4,62152,50,36624,58
148,taylor_sin_decor_32,"(0.79, 2.36)",4.294177e-07,0.00197,24,3,52504,46,34208,56
145,taylor_sin_naive_32,"(0.79, 2.36)",4.294340e-07,0.00022,6,7,48336,40,31416,26
35,fourier_sin_32,"(0.79, 2.36)",9.267859e-04,0.00103,2,1,96384,8,26512,4


In [134]:
by_interval(df[df['Method'].str.contains('cos')].sort_values(by='MAE'))

,Method,Interval,MAE,Runtime,Partitions count,Truncations count,Bytes sent CP0,Requests sent CP0,Bytes sent CP1,Requests sent CP1
2,decor_cos_32,"(-1.0, 2.5)",4.548393e-10,0.00005,2,1,9664,8,4832,4
71,chebyshev_cos_naive_32,"(-1.0, 2.5)",6.760316e-07,0.00080,8,10,67664,56,43496,36
64,chebyshev_cos_clenshaw_32,"(-1.0, 2.5)",6.760573e-07,0.00105,8,10,67648,56,43488,36
76,chebyshev_cos_decor_32,"(-1.0, 2.5)",6.777732e-07,0.00321,24,4,62152,50,36624,58
134,taylor_cos_naive_32,"(-1.0, 2.5)",8.799819e-04,0.00020,6,7,48336,40,31416,26
129,taylor_cos_decor_32,"(-1.0, 2.5)",8.799821e-04,0.00212,24,3,52504,46,34208,56
29,fourier_cos_32,"(-1.0, 2.5)",4.844328e-02,0.00132,1,1,96384,8,24096,2


,Method,Interval,MAE,Runtime,Partitions count,Truncations count,Bytes sent CP0,Requests sent CP0,Bytes sent CP1,Requests sent CP1
24,fourier_cos_32,"(-3.14, 3.14)",5.144924e-10,0.00011,1,1,9696,8,2424,2
0,decor_cos_32,"(-3.14, 3.14)",5.353021e-10,0.00006,2,1,9664,8,4832,4
61,chebyshev_cos_decor_32,"(-3.14, 3.14)",2.695268e-05,0.00210,24,4,62152,50,36624,58
49,chebyshev_cos_clenshaw_32,"(-3.14, 3.14)",2.695364e-05,0.00032,8,10,67648,56,43488,36
56,chebyshev_cos_naive_32,"(-3.14, 3.14)",2.695369e-05,0.00029,8,10,67664,56,43496,36
119,taylor_cos_decor_32,"(-3.14, 3.14)",2.587276e-02,0.00149,24,3,52504,46,34208,56
124,taylor_cos_naive_32,"(-3.14, 3.14)",2.587297e-02,0.00021,6,7,48336,40,31416,26


,Method,Interval,MAE,Runtime,Partitions count,Truncations count,Bytes sent CP0,Requests sent CP0,Bytes sent CP1,Requests sent CP1
6,decor_cos_32,"(-5.0, 5.0)",5.002287e-10,0.00006,2,1,9664,8,4832,4
110,chebyshev_cos_naive_32,"(-5.0, 5.0)",2.028893e-03,0.00073,8,10,67664,56,43496,36
103,chebyshev_cos_clenshaw_32,"(-5.0, 5.0)",2.028893e-03,0.00086,8,10,67648,56,43488,36
115,chebyshev_cos_decor_32,"(-5.0, 5.0)",2.028916e-03,0.00326,24,4,62152,50,36624,58
42,fourier_cos_32,"(-5.0, 5.0)",8.133451e-03,0.00133,1,1,96384,8,24096,2
155,taylor_cos_decor_32,"(-5.0, 5.0)",9.343856e-01,0.00211,24,3,52504,46,34208,56
160,taylor_cos_naive_32,"(-5.0, 5.0)",9.343927e-01,0.00059,6,7,48336,40,31416,26


,Method,Interval,MAE,Runtime,Partitions count,Truncations count,Bytes sent CP0,Requests sent CP0,Bytes sent CP1,Requests sent CP1
4,decor_cos_32,"(0.79, 2.36)",6.831179e-10,0.00006,2,1,9664,8,4832,4
93,chebyshev_cos_naive_32,"(0.79, 2.36)",8.162591e-10,0.00119,8,10,67664,56,43496,36
79,chebyshev_cos_clenshaw_32,"(0.79, 2.36)",9.209119e-10,0.00067,8,10,67648,56,43488,36
100,chebyshev_cos_decor_32,"(0.79, 2.36)",3.660718e-09,0.00204,24,4,62152,50,36624,58
146,taylor_cos_decor_32,"(0.79, 2.36)",4.866597e-06,0.00184,24,3,52504,46,34208,56
149,taylor_cos_naive_32,"(0.79, 2.36)",4.866612e-06,0.00020,6,7,48336,40,31416,26
34,fourier_cos_32,"(0.79, 2.36)",5.095119e-02,0.00128,1,1,96384,8,24096,2


In [135]:
by_interval(df[df['Method'].str.contains('exp')].sort_values(by='MAE'))

,Method,Interval,MAE,Runtime,Partitions count,Truncations count,Bytes sent CP0,Requests sent CP0,Bytes sent CP1,Requests sent CP1
13,decor_exp_32,"(-1.0, 2.5)",2.428502e-09,0.00101,24,3,38048,46,34208,56
65,chebyshev_exp_decor_32,"(-1.0, 2.5)",2.423007e-06,0.00238,24,4,62152,50,36624,58
75,chebyshev_exp_naive_32,"(-1.0, 2.5)",2.426092e-06,0.00197,8,10,67664,56,43496,36
62,chebyshev_exp_clenshaw_32,"(-1.0, 2.5)",2.426097e-06,0.00035,8,10,67648,56,43488,36
130,taylor_exp_naive_32,"(-1.0, 2.5)",2.929406e-03,0.00093,6,7,48336,40,31416,26
135,taylor_exp_decor_32,"(-1.0, 2.5)",2.929408e-03,0.00222,24,3,52504,46,34208,56
33,fourier_exp_32,"(-1.0, 2.5)",4.253193e-01,0.00130,1,1,96384,8,24096,2


,Method,Interval,MAE,Runtime,Partitions count,Truncations count,Bytes sent CP0,Requests sent CP0,Bytes sent CP1,Requests sent CP1
9,decor_exp_32,"(-3.14, 3.14)",2.631791e-08,0.00105,24,3,38048,46,34208,56
47,chebyshev_exp_clenshaw_32,"(-3.14, 3.14)",2.600736e-04,0.00139,8,10,67648,56,43488,36
60,chebyshev_exp_naive_32,"(-3.14, 3.14)",2.600739e-04,0.00150,8,10,67664,56,43496,36
50,chebyshev_exp_decor_32,"(-3.14, 3.14)",2.600750e-04,0.00261,24,4,62152,50,36624,58
120,taylor_exp_naive_32,"(-3.14, 3.14)",8.999560e-02,0.00019,6,7,48336,40,31416,26
125,taylor_exp_decor_32,"(-3.14, 3.14)",8.999568e-02,0.00195,24,3,52504,46,34208,56
28,fourier_exp_32,"(-3.14, 3.14)",8.301076e-01,0.00034,1,1,96384,8,24096,2


,Method,Interval,MAE,Runtime,Partitions count,Truncations count,Bytes sent CP0,Requests sent CP0,Bytes sent CP1,Requests sent CP1
21,decor_exp_32,"(-5.0, 5.0)",4.722351e-07,0.00178,24,3,38048,46,34208,56
104,chebyshev_exp_decor_32,"(-5.0, 5.0)",2.406768e-02,0.00284,24,4,62152,50,36624,58
101,chebyshev_exp_clenshaw_32,"(-5.0, 5.0)",2.406776e-02,0.00031,8,10,67648,56,43488,36
114,chebyshev_exp_naive_32,"(-5.0, 5.0)",2.406777e-02,0.00127,8,10,67664,56,43496,36
156,taylor_exp_naive_32,"(-5.0, 5.0)",2.796562e+00,0.00094,6,7,48336,40,31416,26
161,taylor_exp_decor_32,"(-5.0, 5.0)",2.796563e+00,0.00174,24,3,52504,46,34208,56
46,fourier_exp_32,"(-5.0, 5.0)",5.321508e+00,0.00185,1,1,96384,8,24096,2


,Method,Interval,MAE,Runtime,Partitions count,Truncations count,Bytes sent CP0,Requests sent CP0,Bytes sent CP1,Requests sent CP1
17,decor_exp_32,"(0.79, 2.36)",1.571152e-09,0.00116,24,3,38048,46,34208,56
77,chebyshev_exp_clenshaw_32,"(0.79, 2.36)",3.973780e-09,0.00196,8,10,67648,56,43488,36
99,chebyshev_exp_naive_32,"(0.79, 2.36)",4.141020e-09,0.00068,8,10,67664,56,43496,36
82,chebyshev_exp_decor_32,"(0.79, 2.36)",6.371233e-09,0.00336,24,4,62152,50,36624,58
139,taylor_exp_naive_32,"(0.79, 2.36)",2.374045e-05,0.00021,6,7,48336,40,31416,26
144,taylor_exp_decor_32,"(0.79, 2.36)",2.374145e-05,0.00184,24,3,52504,46,34208,56
38,fourier_exp_32,"(0.79, 2.36)",3.009865e-01,0.00175,1,1,96384,8,24096,2


In [136]:
by_interval(df[df['Method'].str.contains('sigmoid')].sort_values(by='MAE'))

,Method,Interval,MAE,Runtime,Partitions count,Truncations count,Bytes sent CP0,Requests sent CP0,Bytes sent CP1,Requests sent CP1
12,decor_sigmoid_32,"(-1.0, 2.5)",6.851102e-10,0.00651,186,25,282048,346,277216,432
63,chebyshev_sigmoid_decor_32,"(-1.0, 2.5)",1.556778e-06,0.00336,24,4,62152,50,36624,58
67,chebyshev_sigmoid_naive_32,"(-1.0, 2.5)",1.557527e-06,0.00063,8,10,67664,56,43496,36
73,chebyshev_sigmoid_clenshaw_32,"(-1.0, 2.5)",1.557609e-06,0.00127,8,10,67648,56,43488,36
126,taylor_sigmoid_decor_32,"(-1.0, 2.5)",8.343982e-03,0.00191,24,3,45280,46,34208,56
127,taylor_sigmoid_naive_32,"(-1.0, 2.5)",8.343982e-03,0.00011,3,4,26592,22,16920,14
31,fourier_sigmoid_32,"(-1.0, 2.5)",2.360811e-02,0.00034,1,1,96384,8,24096,2


,Method,Interval,MAE,Runtime,Partitions count,Truncations count,Bytes sent CP0,Requests sent CP0,Bytes sent CP1,Requests sent CP1
8,decor_sigmoid_32,"(-3.14, 3.14)",5.031251e-09,0.00622,186,25,282048,346,277216,432
48,chebyshev_sigmoid_decor_32,"(-3.14, 3.14)",2.293994e-04,0.00357,24,4,62152,50,36624,58
52,chebyshev_sigmoid_naive_32,"(-3.14, 3.14)",2.293995e-04,0.00029,8,10,67664,56,43496,36
58,chebyshev_sigmoid_clenshaw_32,"(-3.14, 3.14)",2.293995e-04,0.00113,8,10,67648,56,43488,36
117,taylor_sigmoid_naive_32,"(-3.14, 3.14)",2.176966e-02,0.00042,3,4,26592,22,16920,14
116,taylor_sigmoid_decor_32,"(-3.14, 3.14)",2.176966e-02,0.00069,24,3,45280,46,34208,56
26,fourier_sigmoid_32,"(-3.14, 3.14)",3.304574e-02,0.00173,1,1,96384,8,24096,2


,Method,Interval,MAE,Runtime,Partitions count,Truncations count,Bytes sent CP0,Requests sent CP0,Bytes sent CP1,Requests sent CP1
20,decor_sigmoid_32,"(-5.0, 5.0)",9.631158e-08,0.00758,186,25,282048,346,277216,432
102,chebyshev_sigmoid_decor_32,"(-5.0, 5.0)",2.476074e-03,0.00296,24,4,62152,50,36624,58
106,chebyshev_sigmoid_naive_32,"(-5.0, 5.0)",2.476077e-03,0.00075,8,10,67664,56,43496,36
112,chebyshev_sigmoid_clenshaw_32,"(-5.0, 5.0)",2.476077e-03,0.00183,8,10,67648,56,43488,36
44,fourier_sigmoid_32,"(-5.0, 5.0)",3.554065e-02,0.00034,1,1,96384,8,24096,2
153,taylor_sigmoid_naive_32,"(-5.0, 5.0)",2.192391e-01,0.00012,3,4,26592,22,16920,14
152,taylor_sigmoid_decor_32,"(-5.0, 5.0)",2.192391e-01,0.00111,24,3,45280,46,34208,56


,Method,Interval,MAE,Runtime,Partitions count,Truncations count,Bytes sent CP0,Requests sent CP0,Bytes sent CP1,Requests sent CP1
16,decor_sigmoid_32,"(0.79, 2.36)",2.116667e-10,0.00808,186,25,282048,346,277216,432
85,chebyshev_sigmoid_naive_32,"(0.79, 2.36)",3.557856e-10,0.00118,8,10,67664,56,43496,36
96,chebyshev_sigmoid_clenshaw_32,"(0.79, 2.36)",4.936508e-10,0.00233,8,10,67648,56,43488,36
78,chebyshev_sigmoid_decor_32,"(0.79, 2.36)",3.151028e-09,0.00237,24,4,62152,50,36624,58
136,taylor_sigmoid_decor_32,"(0.79, 2.36)",2.209329e-04,0.00213,24,3,45280,46,34208,56
137,taylor_sigmoid_naive_32,"(0.79, 2.36)",2.209329e-04,0.00099,3,4,26592,22,16920,14
36,fourier_sigmoid_32,"(0.79, 2.36)",8.161617e-03,0.00079,1,1,96384,8,24096,2


In [137]:
by_interval(df[df['Method'].str.contains('sqrt')].sort_values(by='MAE'))

,Method,Interval,MAE,Runtime,Partitions count,Truncations count,Bytes sent CP0,Requests sent CP0,Bytes sent CP1,Requests sent CP1
98,chebyshev_sqrt_clenshaw_32,"(0.79, 2.36)",1.134100e-07,0.00106,8,10,67648,56,43488,36
94,chebyshev_sqrt_naive_32,"(0.79, 2.36)",1.134333e-07,0.00031,8,10,67664,56,43496,36
88,chebyshev_sqrt_decor_32,"(0.79, 2.36)",1.134566e-07,0.00239,24,4,62152,50,36624,58
140,taylor_sqrt_naive_32,"(0.79, 2.36)",8.709681e-03,0.00023,6,7,48336,40,31416,26
142,taylor_sqrt_decor_32,"(0.79, 2.36)",8.709681e-03,0.00220,24,3,52504,46,34208,56
41,fourier_sqrt_32,"(0.79, 2.36)",2.336460e-02,0.00125,1,1,96384,8,24096,2


In [138]:
by_interval(df[df['Method'].str.contains('mul_inv')].sort_values(by='MAE'))

,Method,Interval,MAE,Runtime,Partitions count,Truncations count,Bytes sent CP0,Requests sent CP0,Bytes sent CP1,Requests sent CP1
91,chebyshev_mul_inv_clenshaw_32,"(0.79, 2.36)",0.000006,0.00154,8,10,67648,56,43488,36
81,chebyshev_mul_inv_naive_32,"(0.79, 2.36)",0.000006,0.00119,8,10,67664,56,43496,36
84,chebyshev_mul_inv_decor_32,"(0.79, 2.36)",0.000006,0.00233,24,4,62152,50,36624,58
150,taylor_mul_inv_decor_32,"(0.79, 2.36)",0.000839,0.00207,24,3,52504,46,34208,56
138,taylor_mul_inv_naive_32,"(0.79, 2.36)",0.000839,0.00196,6,7,48336,40,31416,26
40,fourier_mul_inv_32,"(0.79, 2.36)",0.030547,0.00188,1,1,96384,8,24096,2


In [139]:
by_interval(df[df['Method'].str.contains('log')].sort_values(by='MAE'))

,Method,Interval,MAE,Runtime,Partitions count,Truncations count,Bytes sent CP0,Requests sent CP0,Bytes sent CP1,Requests sent CP1
80,chebyshev_log_decor_32,"(0.79, 2.36)",9.808542e-07,0.00322,24,4,62152,50,36624,58
87,chebyshev_log_clenshaw_32,"(0.79, 2.36)",9.810079e-07,0.00096,8,10,67648,56,43488,36
86,chebyshev_log_naive_32,"(0.79, 2.36)",9.810312e-07,0.00066,8,10,67664,56,43496,36
151,taylor_log_decor_32,"(0.79, 2.36)",1.783265e-04,0.00208,24,3,52504,46,34208,56
141,taylor_log_naive_32,"(0.79, 2.36)",1.783266e-04,0.00021,6,7,48336,40,31416,26
39,fourier_log_32,"(0.79, 2.36)",3.955955e-02,0.00127,1,1,96384,8,24096,2


In [140]:
by_interval(df[df['Method'].str.contains('tanh')].sort_values(by='MAE'))

,Method,Interval,MAE,Runtime,Partitions count,Truncations count,Bytes sent CP0,Requests sent CP0,Bytes sent CP1,Requests sent CP1
14,decor_tanh_32,"(-1.0, 2.5)",3.628155e-09,0.14371,184,59,3505632,534,1986528,496
69,chebyshev_tanh_decor_32,"(-1.0, 2.5)",6.157419e-04,0.00238,24,4,62152,50,36624,58
68,chebyshev_tanh_naive_32,"(-1.0, 2.5)",6.157422e-04,0.00121,8,10,67664,56,43496,36
74,chebyshev_tanh_clenshaw_32,"(-1.0, 2.5)",6.157422e-04,0.00030,8,10,67648,56,43488,36
32,fourier_tanh_32,"(-1.0, 2.5)",6.300809e-02,0.00160,1,1,96384,8,24096,2
131,taylor_tanh_naive_32,"(-1.0, 2.5)",1.508766e-01,0.00013,3,4,26592,22,16920,14
132,taylor_tanh_decor_32,"(-1.0, 2.5)",1.508767e-01,0.00109,24,3,45280,46,34208,56


,Method,Interval,MAE,Runtime,Partitions count,Truncations count,Bytes sent CP0,Requests sent CP0,Bytes sent CP1,Requests sent CP1
10,decor_tanh_32,"(-3.14, 3.14)",2.703032e-07,0.13681,184,59,3505632,534,1986528,496
53,chebyshev_tanh_naive_32,"(-3.14, 3.14)",1.179839e-02,0.00148,8,10,67664,56,43496,36
59,chebyshev_tanh_clenshaw_32,"(-3.14, 3.14)",1.179839e-02,0.00127,8,10,67648,56,43488,36
54,chebyshev_tanh_decor_32,"(-3.14, 3.14)",1.179841e-02,0.00235,24,4,62152,50,36624,58
27,fourier_tanh_32,"(-3.14, 3.14)",7.177069e-02,0.00033,1,1,96384,8,24096,2
122,taylor_tanh_decor_32,"(-3.14, 3.14)",1.857928e+00,0.00107,24,3,45280,46,34208,56
121,taylor_tanh_naive_32,"(-3.14, 3.14)",1.857928e+00,0.00012,3,4,26592,22,16920,14


,Method,Interval,MAE,Runtime,Partitions count,Truncations count,Bytes sent CP0,Requests sent CP0,Bytes sent CP1,Requests sent CP1
22,decor_tanh_32,"(-5.0, 5.0)",0.001147,0.14265,184,59,3505632,534,1986528,496
108,chebyshev_tanh_decor_32,"(-5.0, 5.0)",0.039980,0.00239,24,4,62152,50,36624,58
107,chebyshev_tanh_naive_32,"(-5.0, 5.0)",0.039980,0.00120,8,10,67664,56,43496,36
113,chebyshev_tanh_clenshaw_32,"(-5.0, 5.0)",0.039980,0.00035,8,10,67648,56,43488,36
45,fourier_tanh_32,"(-5.0, 5.0)",0.072037,0.00035,1,1,96384,8,24096,2
158,taylor_tanh_decor_32,"(-5.0, 5.0)",9.070970,0.00064,24,3,45280,46,34208,56
157,taylor_tanh_naive_32,"(-5.0, 5.0)",9.070970,0.00012,3,4,26592,22,16920,14


,Method,Interval,MAE,Runtime,Partitions count,Truncations count,Bytes sent CP0,Requests sent CP0,Bytes sent CP1,Requests sent CP1
18,decor_tanh_32,"(0.79, 2.36)",2.627309e-10,0.14359,184,59,3505632,534,1986528,496
89,chebyshev_tanh_naive_32,"(0.79, 2.36)",6.107772e-08,0.00029,8,10,67664,56,43496,36
97,chebyshev_tanh_clenshaw_32,"(0.79, 2.36)",6.108703e-08,0.00117,8,10,67648,56,43488,36
90,chebyshev_tanh_decor_32,"(0.79, 2.36)",6.267577e-08,0.00229,24,4,62152,50,36624,58
147,taylor_tanh_naive_32,"(0.79, 2.36)",1.808822e-03,0.00061,3,4,26592,22,16920,14
143,taylor_tanh_decor_32,"(0.79, 2.36)",1.808822e-03,0.00182,24,3,45280,46,34208,56
37,fourier_tanh_32,"(0.79, 2.36)",1.175506e-02,0.00033,1,1,96384,8,24096,2


In [141]:
by_interval(df[df['Method'].str.contains('polynomial')].sort_values(by='MAE'))

,Method,Interval,MAE,Runtime,Partitions count,Truncations count,Bytes sent CP0,Requests sent CP0,Bytes sent CP1,Requests sent CP1
15,decor_polynomial_32,"(-1.0, 2.5)",1.491446e-08,0.0019,24,3,45280,46,34208,56


,Method,Interval,MAE,Runtime,Partitions count,Truncations count,Bytes sent CP0,Requests sent CP0,Bytes sent CP1,Requests sent CP1
11,decor_polynomial_32,"(-3.14, 3.14)",1.521215e-07,0.00138,24,3,45280,46,34208,56


,Method,Interval,MAE,Runtime,Partitions count,Truncations count,Bytes sent CP0,Requests sent CP0,Bytes sent CP1,Requests sent CP1
23,decor_polynomial_32,"(-5.0, 5.0)",8.007355e-07,0.00195,24,3,45280,46,34208,56


,Method,Interval,MAE,Runtime,Partitions count,Truncations count,Bytes sent CP0,Requests sent CP0,Bytes sent CP1,Requests sent CP1
19,decor_polynomial_32,"(0.79, 2.36)",1.259862e-08,0.00198,24,3,45280,46,34208,56


In [142]:
""" E2E apps """

# Until Codon Jupyter is fixed: Read the data from files
dump_folder = "dump"
dump_files = [
    "log_reg",
    "dti"
    ]
nbit_fs = [32]

df_data = {
    'Method': [],
    'Accuracy': [],
    'Runtime': [],
    'Bytes sent': [],
    'Requests sent': [],
    'Partitions count': [],
    'Truncations count': []
    }

for dump_file in dump_files:
    for nbit_f in nbit_fs:
        try:
            with open(f"{dump_folder}/{dump_file}_{nbit_f}.p", "rb") as f:
                data = pickle.load(f)
                for k, v in data.items():
                    if not k.endswith('_accuracy'):
                        continue
                    
                    k = k.replace('_accuracy', '')
                    accuracy = data[f"{k}_accuracy"][0]
                    runtime = round(data[f"{k}_time"][0], 5)
                    bytes_sent = int(data[f"{k}_bytes_sent"][0])
                    send_requests = int(data[f"{k}_send_requests"][0])
                    partitions_count = int(data[f"{k}_partitions_count"][0])
                    truncations_count = int(data[f"{k}_truncations_count"][0])
                    
                    df_data['Method'].append(f"{k}_{nbit_f}")
                    
                    df_data['Accuracy'].append(accuracy)
                    df_data['Runtime'].append(runtime)
                    df_data['Bytes sent'].append(bytes_sent)
                    df_data['Requests sent'].append(send_requests)
                    df_data['Partitions count'].append(partitions_count)
                    df_data['Truncations count'].append(truncations_count)
        except FileNotFoundError:
            print(f"Could not find {dump_folder}/{dump_file}_{nbit_f}.p")

e2e_df = pd.DataFrame(df_data)

In [143]:
e2e_df[e2e_df['Method'].str.contains('binary')].sort_values(by='Accuracy', ascending=False)

,Method,Accuracy,Runtime,Bytes sent,Requests sent,Partitions count,Truncations count
4,log_reg_binary_decor_32,0.9375,0.21557,1237072,6602,1841,1440
1,log_reg_binary_chebyshev_32,0.8750,0.03795,205392,922,201,260
2,log_reg_binary_fourier_32,0.6875,0.19390,148112,282,81,80
5,log_reg_binary_taylor_32,0.5000,0.02244,117552,482,101,140


In [144]:
e2e_df[e2e_df['Method'].str.contains('multinomial')].sort_values(by='Accuracy', ascending=False)

,Method,Accuracy,Runtime,Bytes sent,Requests sent,Partitions count,Truncations count
3,log_reg_multinomial_fourier_32,0.9375,0.56319,2458512,5122,2361,80
6,log_reg_multinomial_chebyshev_32,0.9375,0.40785,2623312,5762,2481,260
7,log_reg_multinomial_taylor_32,0.9375,0.39607,2506672,5562,2441,200
0,log_reg_multinomial_decor_32,0.7500,0.40416,2537552,6162,2801,120


In [145]:
e2e_df[e2e_df['Method'].str.contains('dti')].sort_values(by='Accuracy', ascending=False)

,Method,Accuracy,Runtime,Bytes sent,Requests sent,Partitions count,Truncations count
15,dti_linear_chebyshev_32,0.708333,168.88176,2403710984,5646,2201,498
8,dti_tanh_chebyshev_32,0.583333,169.70991,2372641816,5876,1885,969
13,dti_sigmoid_fourier_32,0.562500,178.18849,2401271016,9124,3299,1180
9,dti_tanh_fourier_32,0.520833,173.81166,2371618536,4564,1639,600
10,dti_tanh_decor_32,0.520833,182.78921,2490082072,24736,9101,2978
11,dti_tanh_taylor_32,0.500000,173.72765,2371038256,4974,1680,723
12,dti_sigmoid_chebyshev_32,0.500000,176.92023,2402294296,10436,3545,1549
14,dti_sigmoid_taylor_32,0.500000,177.04375,2400690736,9534,3340,1303
16,dti_sigmoid_decor_32,0.479167,187.79492,2519734552,29296,10761,3558
